In [1]:
import sys
import os
import cv2
import numpy as np
import torch
from torch.nn import functional as F

sys.path.append(os.path.dirname(os.getcwd()))
from model.baselines.dunet import DUNet
from utils.transform import validation_transform
from utils.datasets import CropDatasetWithLabel, CropDatasetNoLabel

/home/zhenyi/anaconda3/envs/fundusegmenter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# True for image + mask, False for image only
with_label = True
# set as true if dataset is IDRID for 1200*1200 cropping
if_idrid = False
if_vampire_train = False
checkpoint_path = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/DUNet_OD_CentreCrop_pretrained.pth'
image_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Vampire/testing/images'
mask_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Vampire/testing/masks'
output_base_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Vampire/testing'
# Threshold for area check
min_area_ratio = 0.01
max_area_ratio = 0.3

In [3]:
# centre cropping
def centre_crop(centre, original_hw, image_original, if_idrid):
    x, y = centre
    h, w = original_hw

    if if_idrid is False:
        if if_vampire_train is True:
            left = x-200
            right = x+200
            top = y-200
            bottom = y+200
        if if_vampire_train is False:
            left = x-400
            right = x+400
            top = y-400
            bottom = y+400
    if if_idrid is True:
        left = x-600
        right = x+600
        top = y-600
        bottom = y+600

    pad_left = max(0, -left)
    pad_right = max(0, right - w)
    pad_top = max(0, -top)
    pad_bottom = max(0, bottom - h)

    if image_original.dtype == bool:
        image_original = image_original.astype(np.uint8)*255
    
    if image_original.max() <= 1:
        image_original = image_original*255

    if pad_left > 0 or pad_right > 0 or pad_top > 0 or pad_bottom > 0:

        if len(np.unique(image_original)) == 3:
            padded_image = cv2.copyMakeBorder(image_original, 
                                              pad_top, pad_bottom, pad_left, pad_right,
                                              cv2.BORDER_CONSTANT, 
                                              value=255,
                                             )
        else:                
            padded_image = cv2.copyMakeBorder(image_original, 
                                              pad_top, pad_bottom, pad_left, pad_right,
                                              cv2.BORDER_CONSTANT, 
                                              value=0,
                                             )
                
        left += pad_left
        right += pad_left
        top += pad_top
        bottom += pad_top
    else:
        padded_image = image_original
    
    cropped_region = padded_image[top:bottom, left:right]

    return cropped_region

In [4]:
# load DUNet
model = DUNet(input_channel=3, output_channel=2, base_channel=32)
    
# load weights
checkpoint = torch.load(checkpoint_path, map_location='cpu')
checkpoint_model = checkpoint['model']
msg = model.load_state_dict(checkpoint_model, strict=True)

transform = validation_transform(image_size=256)

if with_label is True:
    dataset = CropDatasetWithLabel(image_root_path = image_dir, 
                                   mask_root_path = mask_dir, 
                                   transform = transform,
                                  )
    os.makedirs(os.path.join(output_base_dir, 'ROIs/images'), exist_ok=True)
    os.makedirs(os.path.join(output_base_dir, 'ROIs/masks'), exist_ok=True)
if with_label is False:
    dataset = CropDatasetNoLabel(image_root_path = image_dir, 
                                 transform = transform,
                                )
    os.makedirs(os.path.join(output_base_dir, 'ROIs/images'), exist_ok=True)

centres_file_path = os.path.join(output_base_dir, 'ROIs', 'centres.txt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

with open(centres_file_path, 'w') as centres_file:
    centres_file.write("# filename,centre_x,centre_y,original_height,original_width\n")

    with torch.no_grad():   
        for i in dataset:
            if with_label is True:
                x, mask, image_original, filename = i[0].unsqueeze(dim=0), i[1], i[2], i[3]
            if with_label is False:
                x, image_original, filename = i[0].unsqueeze(dim=0), i[1], i[2]
            x = x.to(device)
            y_pred = model(x)
            y_pred = F.interpolate(y_pred, size=(image_original.shape[0], image_original.shape[1]), mode='bicubic')
            y_pred = torch.argmax(torch.softmax(y_pred,1), dim=1) 

            # find all the contours
            y_pred = y_pred.squeeze().cpu().numpy().astype(np.uint8)
            if y_pred.max() <= 1:
                y_pred = (y_pred * 255).astype(np.uint8)                
            contours, _ = cv2.findContours(y_pred, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # remove contours by area check and circularity check
            best_circularity = -1
            best_contour = None
            for contour in contours:
                area = cv2.contourArea(contour)
                area_ratio = area / (image_original.shape[0]*image_original.shape[1])      
                if area_ratio < min_area_ratio or area_ratio > max_area_ratio:
                    continue
                perimeter = cv2.arcLength(contour, True)
                if perimeter == 0:
                    circularity = 0
                else:
                    circularity = 4 * np.pi * area / (perimeter * perimeter)
                if circularity > best_circularity:
                    best_circularity = circularity
                    best_contour = contour
            if best_contour is None:
                best_contour = max(contours, key=cv2.contourArea)

            # find centre
            (x, y), radius = cv2.minEnclosingCircle(best_contour)            
            centre = (int(x), int(y))                   
            (h, w) = image_original.shape[:2]

            # write centre file
            centres_file.write(f"{filename},{centre[0]},{centre[1]},{h},{w}\n")
        
            if with_label is True:
                cropped_image = centre_crop(centre, (h, w), image_original, if_idrid)
                cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_RGB2BGR)
                cropped_mask = centre_crop(centre, (h, w), mask, if_idrid)
                cropped_mask = cv2.cvtColor(cropped_mask, cv2.COLOR_GRAY2BGR)
                cv2.imwrite(os.path.join(output_base_dir, 'ROIs/images', filename[:-4]+'.png'), cropped_image, [cv2.IMWRITE_PNG_COMPRESSION, 0])
                cv2.imwrite(os.path.join(output_base_dir, 'ROIs/masks', filename[:-4]+'.png'), cropped_mask, [cv2.IMWRITE_PNG_COMPRESSION, 0])

            if with_label is False:
                cropped_image = centre_crop(centre, (h, w), image_original, if_idrid)
                cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_RGB2BGR)
                cv2.imwrite(os.path.join(output_base_dir, 'ROIs/images', filename[:-4]+'.png'), cropped_image, [cv2.IMWRITE_PNG_COMPRESSION, 0])